# Device Discovery

Discover the quantum devices Amazon Braket exposes and learn to read their properties -- without spending a cent or needing AWS credentials.

**Objectives:**
- Enumerate Braket devices through the `AwsDevice.get_devices()` API (shown, guarded behind a flag)
- Read the fields of a device-properties object: qubit count, native gate set, connectivity, status, queue depth
- Inspect the device-property SCHEMA locally, with no AWS call, so you know what to expect
- Run a real circuit on the credential-free `LocalSimulator`
- Summarize the Braket fleet (IonQ Forte, IQM Garnet, QuEra Aquila, SV1/DM1/TN1) and its key specs

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.

In [ ]:
# Setup: Run this cell first
# Requires: pip install -e '.[dev]' from the project root (see `make setup`)

import numpy as np
import matplotlib.pyplot as plt

from braket.circuits import Circuit
from braket.devices import LocalSimulator

# Importing AwsDevice is free and touches no network. We will NOT instantiate it
# or call get_devices() at top level -- every line that would reach AWS is guarded
# below behind `if RUN_ON_AWS:`.
from braket.aws import AwsDevice

# Master switch for anything that needs AWS credentials and may incur cost.
# Left False so this notebook runs end-to-end offline. Set True only with
# credentials configured (`make setup`) and after reading the cost note in each guard.
RUN_ON_AWS = False

# The local state-vector simulator needs no AWS account. It is your default
# for development -- free, instant, exact up to ~25 qubits.
local = LocalSimulator()
print("Local simulator ready:", local.name)
print("RUN_ON_AWS =", RUN_ON_AWS, "(AWS calls are disabled)")

## 1. Something real, locally: a Bell pair

Before we talk about hardware you cannot reach without credentials, run a circuit on hardware you
can -- your own laptop. The `LocalSimulator` executes the Bell pair from `01-foundations` exactly,
at zero cost. This is the bottom rung of the simulator ladder, and the rung you live on while
developing. Only the correlated outcomes |00> and |11> appear, because the CNOT entangles the two
qubits after the Hadamard puts qubit 0 into superposition.

In [ ]:
# Build and run a Bell pair on the local simulator (no AWS, no cost).
bell = Circuit().h(0).cnot(0, 1)
print(bell)
print(f"\nqubit_count = {bell.qubit_count}, depth = {bell.depth}")

result = local.run(bell, shots=1000).result()
counts = dict(result.measurement_counts)
print("\nMeasurement counts:", counts)

# An entangled Bell pair only ever yields the correlated bitstrings.
assert set(counts).issubset({"00", "11"}), "Bell pair must only produce |00> and |11>"
assert sum(counts.values()) == 1000, "Braket returns exactly the requested shot count"
assert bell.qubit_count == 2 and bell.depth == 2
print("\nOK: only |00> and |11> appear -- the qubits are perfectly correlated.")

## 2. The discovery API (shown, not called)

With credentials, you would list the whole fleet and read any device's live properties like this:

```python
# Enumerate every device Braket exposes to your account/region:
all_devices = AwsDevice.get_devices()
for d in all_devices:
    print(d.name, d.provider_name, d.status)

# Inspect one device by ARN:
forte = AwsDevice("arn:aws:braket:us-east-1::device/qpu/ionq/Forte-1")
print(forte.status)                       # ONLINE / OFFLINE / RETIRED
print(forte.properties.paradigm.qubitCount)
print(forte.properties.paradigm.nativeGateSet)
print(forte.properties.paradigm.connectivity.fullyConnected)
print(forte.queue_depth())                # how many tasks are ahead of yours
```

Every one of those calls reaches AWS (and `AwsDevice(arn)` for a managed device fetches its
properties over the network), so we keep the live demonstration behind `RUN_ON_AWS`. The cell
below performs **no** AWS work when the flag is False; it only records the ARNs you would target.
Querying device metadata is itself free, but it requires credentials -- so we still gate it.

In [ ]:
# Device ARNs are just strings -- defining them touches no network and constructs no AwsDevice.
# (These are exactly what the braket.devices.Devices constants resolve to.)
DEVICE_ARNS = {
    "IonQ Forte-1":  "arn:aws:braket:us-east-1::device/qpu/ionq/Forte-1",
    "IQM Garnet":    "arn:aws:braket:eu-north-1::device/qpu/iqm/Garnet",
    "QuEra Aquila":  "arn:aws:braket:us-east-1::device/qpu/quera/Aquila",
    "SV1":           "arn:aws:braket:::device/quantum-simulator/amazon/sv1",
    "DM1":           "arn:aws:braket:::device/quantum-simulator/amazon/dm1",
    "TN1":           "arn:aws:braket:::device/quantum-simulator/amazon/tn1",
}
for label, arn in DEVICE_ARNS.items():
    assert arn.startswith("arn:aws:braket:")
    print(f"{label:14s} -> {arn}")

# --- LIVE DISCOVERY (guarded) -----------------------------------------------
# Cost note: get_devices() and reading properties are free, but require AWS
# credentials. Running circuits on these devices is NOT free -- QPUs charge
# ~$0.30/task + per-shot. We do not run anything here; we only read metadata.
if RUN_ON_AWS:
    all_devices = AwsDevice.get_devices()
    print(f"\nBraket exposes {len(all_devices)} device(s) to this account/region:")
    for d in all_devices:
        print(f"  {d.name:24s} {d.provider_name:10s} status={d.status}")

    forte = AwsDevice(DEVICE_ARNS["IonQ Forte-1"])
    para = forte.properties.paradigm
    print("\nIonQ Forte-1 live properties:")
    print("  qubitCount     :", para.qubitCount)
    print("  nativeGateSet  :", para.nativeGateSet)
    print("  fullyConnected :", para.connectivity.fullyConnected)
    print("  queue_depth    :", forte.queue_depth())
else:
    print("\n[RUN_ON_AWS is False] Skipping all live AWS calls.")

## 3. The shape of a device-properties object (offline)

You do not need a live device to learn *what fields a device exposes*. Braket ships the property
schemas as ordinary Python classes in `braket.device_schema`. Importing one and reading its fields
is pure local reflection -- no AWS, no credentials. This tells you exactly what
`device.properties` will contain when you do query a real device, so the live call holds no
surprises.

The field that matters most for hardware reasoning lives under `paradigm`: `qubitCount`,
`nativeGateSet`, and `connectivity` (whose `fullyConnected` flag is the all-to-all-vs-lattice
distinction from the GUIDE -- the SWAP-tax decider).

In [ ]:
# These are schema (pydantic model) classes, not live devices -- importing and
# reading their fields reaches no network.
from braket.device_schema.ionq import IonqDeviceCapabilities
from braket.device_schema.iqm import IqmDeviceCapabilities
from braket.device_schema.gate_model_qpu_paradigm_properties_v1 import (
    GateModelQpuParadigmProperties,
)
from braket.device_schema.device_connectivity import DeviceConnectivity

top_level = list(IonqDeviceCapabilities.__fields__.keys())
paradigm_fields = list(GateModelQpuParadigmProperties.__fields__.keys())
connectivity_fields = list(DeviceConnectivity.__fields__.keys())

print("Top-level fields of a gate-model device-properties object:")
for f in top_level:
    print("  -", f)

print("\nFields under .paradigm (the hardware specs you care about):")
for f in paradigm_fields:
    print("  -", f)

print("\nFields under .paradigm.connectivity:")
for f in connectivity_fields:
    print("  -", f)

# The schema guarantees these exist on any gate-model device's properties.
assert "paradigm" in top_level
assert "qubitCount" in paradigm_fields and "nativeGateSet" in paradigm_fields
assert "fullyConnected" in connectivity_fields
# IonQ and IQM share the gate-model paradigm schema -- same shape, different values.
assert list(IqmDeviceCapabilities.__fields__.keys()) == top_level
print("\nOK: every gate-model device exposes paradigm.qubitCount, .nativeGateSet, .connectivity")

## 4. The Braket fleet at a glance (sourced from the GUIDE)

Reading properties one device at a time is how you confirm specs live. To *choose* a device you
want the whole fleet side by side. The table below is **hardcoded from `../GUIDE.md`** (it does not
query AWS) so it works offline and stays stable. It captures the five trade-off axes from the
GUIDE -- connectivity, fidelity, coherence, clock speed, qubit count -- plus status and the cost
model.

Note the families: IonQ is trapped-ion (all-to-all, slow, high fidelity), IQM is superconducting
(nearest-neighbor lattice, nanosecond gates -- the SWAP tax applies), and QuEra Aquila is the lone
non-gate-model row (neutral-atom analog: you place atoms and drive a Hamiltonian, you do not run
gate circuits).

In [ ]:
# Hardcoded from ../GUIDE.md ("The three hardware families" + "The simulator ladder").
# Values are illustrative spec figures quoted from the GUIDE, not a live query.
DEVICES = [
    # QPUs
    {"name": "IonQ Forte",   "family": "Trapped ion",     "qubits": 36,
     "connectivity": "all-to-all",      "native_gates": "GPi, GPi2, MS",
     "clock": "microseconds", "gate_model": True},
    {"name": "IQM Garnet",   "family": "Superconducting", "qubits": 20,
     "connectivity": "square lattice",  "native_gates": "CZ, PRx",
     "clock": "nanoseconds",  "gate_model": True},
    {"name": "QuEra Aquila", "family": "Neutral atom",    "qubits": 256,
     "connectivity": "geometric (analog)", "native_gates": "-- (analog)",
     "clock": "analog evolution", "gate_model": False},
    # Managed simulators (the simulator ladder)
    {"name": "SV1", "family": "Simulator (state vector)",   "qubits": 34,
     "connectivity": "all-to-all", "native_gates": "all", "clock": "$0.075/min",
     "gate_model": True},
    {"name": "DM1", "family": "Simulator (density matrix)", "qubits": 17,
     "connectivity": "all-to-all", "native_gates": "all + noise", "clock": "$0.075/min",
     "gate_model": True},
    {"name": "TN1", "family": "Simulator (tensor network)", "qubits": 50,
     "connectivity": "all-to-all", "native_gates": "all", "clock": "$0.275/min",
     "gate_model": True},
]

header = f"{'Device':<14}{'Family':<28}{'Qubits':>7}  {'Connectivity':<20}{'Native gates':<18}"
print(header)
print("-" * len(header))
for d in DEVICES:
    print(f"{d['name']:<14}{d['family']:<28}{d['qubits']:>7}  "
          f"{d['connectivity']:<20}{d['native_gates']:<18}")

# Sanity checks against the GUIDE's stated numbers.
qpu = [d for d in DEVICES if not d['family'].startswith('Simulator')]
assert max(x['qubits'] for x in qpu) == 256, "Aquila has the most qubits (256) per the GUIDE"
non_gate = [d for d in DEVICES if not d['gate_model']]
assert [d['name'] for d in non_gate] == ["QuEra Aquila"], "Aquila is the only analog (non-gate) device"
print("\nOK: Aquila tops qubit count (256) and is the sole non-gate-model device.")

In [ ]:
# Visualize qubit counts across the fleet (purely local matplotlib).
names = [d["name"] for d in DEVICES]
qubits = [d["qubits"] for d in DEVICES]
is_qpu = [not d["family"].startswith("Simulator") for d in DEVICES]
colors = ["#2563eb" if q else "#94a3b8" for q in is_qpu]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(names, qubits, color=colors)
ax.set_ylabel("Qubit / atom count")
ax.set_title("Braket fleet capacity (blue = QPU, gray = simulator)  [source: GUIDE]")
ax.tick_params(axis="x", rotation=30)
for i, q in enumerate(qubits):
    ax.text(i, q + 3, str(q), ha="center", fontsize=9)
plt.tight_layout()
plt.show()

print("Capacity is only one axis. Aquila's 256 atoms dwarf the gate-model QPUs, but it cannot")
print("run gate circuits at all -- the right device depends on connectivity, fidelity, and your")
print("problem's structure, not qubit count alone.")

## 5. Exercises

Complete the following. They all run offline -- keep `RUN_ON_AWS = False`.

In [ ]:
# Exercise 1: Build a small "fleet report" dict from DEVICES that maps each
# device family to the total qubit/atom count in that family. Print it sorted
# by total descending. (Hint: iterate DEVICES, accumulate into a dict.)

# TODO: Your code here
# totals = {}
# for d in DEVICES:
#     totals[d['family']] = totals.get(d['family'], 0) + d['qubits']
# for fam, n in sorted(totals.items(), key=lambda kv: -kv[1]):
#     print(f"{fam:<28} {n}")

In [ ]:
# Exercise 2: Write a helper choose_device(needs_noise, needs_all_to_all,
# n_qubits) that returns a device NAME from DEVICES following the GUIDE's
# decision flow (Section "Choosing a device"):
#   - needs_noise            -> 'DM1' (only simulator that models noise)
#   - needs_all_to_all QPU   -> 'IonQ Forte' (all-to-all, most qubits)
#   - otherwise large circuit -> 'SV1' if n_qubits <= 34 else 'TN1'
# Test it on a few cases. (No AWS -- this is pure dispatch logic.)

# TODO: Your code here
# def choose_device(needs_noise, needs_all_to_all, n_qubits):
#     ...
# print(choose_device(needs_noise=True,  needs_all_to_all=False, n_qubits=10))
# print(choose_device(needs_noise=False, needs_all_to_all=True,  n_qubits=20))
# print(choose_device(needs_noise=False, needs_all_to_all=False, n_qubits=40))

In [ ]:
# Exercise 3 (requires AWS -- leave guarded): with credentials configured, set
# RUN_ON_AWS = True at the top, then list every ONLINE device and its queue
# depth. Reading metadata is free, but needs an account. Do NOT submit circuits.

# TODO: Your code here
# if RUN_ON_AWS:
#     for d in AwsDevice.get_devices():
#         if d.status == 'ONLINE':
#             dev = AwsDevice(d.arn)
#             print(d.name, 'queue_depth=', dev.queue_depth())

## Summary

- `AwsDevice.get_devices()` lists the fleet and `AwsDevice(arn).properties` reads one device's specs -- both need AWS credentials, so we kept them behind `RUN_ON_AWS` (left False) with a cost note.
- The credential-free `LocalSimulator` ran a real Bell pair, yielding only the correlated |00> and |11> -- your default development rung.
- Device properties have a fixed shape: under `.paradigm` you find `qubitCount`, `nativeGateSet`, and `connectivity.fullyConnected` (the all-to-all-vs-lattice / SWAP-tax flag). You can read that schema offline before ever querying a live device.
- The fleet sorts into three families plus the simulator ladder: IonQ trapped-ion (all-to-all, high fidelity, microsecond gates), IQM superconducting (lattice, nanosecond gates), QuEra Aquila neutral-atom analog (256 atoms, non-gate-model), and SV1 / DM1 (noise) / TN1.
- Choosing a device is about connectivity, fidelity, and problem structure -- not raw qubit count. And per the project rule: develop on Local first, climb the ladder, touch a QPU only when validated and only with a cost estimate.

**Next:** [02-ionq-exploration.ipynb](02-ionq-exploration.ipynb) -- submit a circuit to a trapped-ion machine (or simulate it locally), inspect native-gate decomposition, and weigh shot counts against cost.